# Tactical GNN v3 — 공유 GAT 백본 + 3-Head

**v2 → v3 수정 (시니어 리서처 리뷰 반영)**

| # | Critical 수정 | 변경 |
|---|-------------|------|
| C1 | Head C 단일프레임→시퀀스 | `cat([emb_t, emb_{t-1}, diff])` temporal differencing |
| C2 | Head A 팀소속→시너지 | 포지션 상보성 × 공간 근접도 연속 타겟 |
| C3 | 라벨 실패→랜덤 할당 | `y_mask` 플래그로 Head B loss 마스킹 |

| # | High/Medium 수정 | 변경 |
|---|----------------|------|
| H1 | mean_pool 정보손실 | `mean + max` 결합 (graph_emb 2D) |
| H2 | Negative transfer | 태스크별 adapter layer 추가 |
| H3 | 가상데이터 무의미 | 포메이션↔전술 상관관계 부여 |
| M1 | 정적 loss 가중치 | Uncertainty Weighting (Kendall 2018) |
| M2 | AI Hub 해상도 하드코딩 | JSON 메타데이터에서 해상도 읽기 |
| M3 | 커스텀 콜레이터 충돌 | `Data.__inc__()` 오버라이드 |
| M4 | lr=1e-3 과다 | lr=3e-4, eta_min=1e-5 |
| M5 | 3-layer over-smoothing | 2-layer + residual connection |

In [ ]:
!nvidia-smi

# 사전 빌드 wheel 사용 (소스 빌드 X → 설치 30초 이내)
import torch
TORCH = torch.__version__.split('+')[0]
CUDA = torch.version.cuda.replace('.', '')
!pip install -q torch-geometric
!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{TORCH}+cu{CUDA}.html
!pip install -q pandas numpy matplotlib scikit-learn tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json, time, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from scipy.spatial.distance import cdist
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

PROJECT_DIR = Path('/content/drive/MyDrive/tactical-gnn')
AIHUB_LABEL_DIR = PROJECT_DIR / 'data' / 'aihub' / 'training_labels' / '1.동영상분석'
AIHUB_IMAGE_DIR = PROJECT_DIR / 'data' / 'aihub' / 'training_labels' / '2.이미지분석'
SOCCERNET_DIR = PROJECT_DIR / 'data' / 'soccernet' / 'tracking'
GRAPH_DIR = PROJECT_DIR / 'data' / 'graphs'
MODEL_DIR = PROJECT_DIR / 'models'
for d in [GRAPH_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. 상수 & 전술 체계

In [ ]:
# --- 전술 분류 (AI Hub depth1) ---
TACTIC_CLASSES = {
    '빌드업-크로스형': 0, '빌드업-중앙침투형': 1, '빌드업-측면침투형': 2, '빌드업-롱볼형': 3,
    '세트피스-코너킥': 4, '세트피스-프리킥': 5, '세트피스-스로인': 6, '세트피스-골킥': 7,
    '수비-고위압박': 8, '수비-중위압박': 9, '수비-저위블록': 10,
    '공격전환': 11, '수비전환': 12,
}
NUM_TACTIC_CLASSES = len(TACTIC_CLASSES)
IDX_TO_TACTIC = {v: k for k, v in TACTIC_CLASSES.items()}

# --- 포지션 ---
POSITION_NAMES = ['GK', 'DF', 'MF', 'FW', 'FP']
POS_TO_IDX = {n: i for i, n in enumerate(POSITION_NAMES)}
POS_TO_ONEHOT = {
    name: [1.0 if i == j else 0.0 for j in range(len(POSITION_NAMES))]
    for i, name in enumerate(POSITION_NAMES)
}
POSITION_WEIGHTS = {'GK': 0.6, 'DF': 0.8, 'MF': 1.0, 'FW': 1.2, 'FP': 1.0}

# --- 포지션 상보성 행렬 (도메인 초기값 — 모델에서 nn.Parameter로 학습 정밀화) ---
#          GK   DF   MF   FW   FP
COMPLEMENTARITY_INIT = np.array([
    [0.1, 0.8, 0.4, 0.2, 0.5],  # GK: DF와 높음, FW와 낮음
    [0.8, 0.6, 0.9, 0.5, 0.7],  # DF: MF와 매우 높음
    [0.4, 0.9, 0.5, 0.9, 0.7],  # MF: DF/FW와 높음, MF끼리 보통
    [0.2, 0.5, 0.9, 0.3, 0.6],  # FW: MF와 높음, FW끼리 낮음
    [0.5, 0.7, 0.7, 0.6, 0.5],  # FP: 기본값
], dtype=np.float32)

# 데이터셋 빌드 시 타겟 생성에는 초기값 사용 (COMPLEMENTARITY_INIT)
# 모델 내부에서는 nn.Parameter로 학습하여 데이터 기반 정밀화
COMPLEMENTARITY = COMPLEMENTARITY_INIT  # 타겟 생성용 alias

PITCH_X, PITCH_Y = 105.0, 68.0
NODE_IN_DIM = 11  # x, y, vx, vy, team, ball_dist, GK, DF, MF, FW, FP
EDGE_IN_DIM = 5   # distance, rel_speed, same_team, dx, dy
HIDDEN_DIM = 64

print(f'전술 {NUM_TACTIC_CLASSES}클래스, 포지션 {len(POSITION_NAMES)}종')
print(f'상보성 행렬 초기값 (도메인 지식):')
print(f'  shape: {COMPLEMENTARITY_INIT.shape}')
print(f'  → 모델에서 nn.Parameter로 학습 정밀화됨')

## 2. 데이터 로드

In [ ]:
def load_aihub_strategy_labels(label_dir: Path) -> list[dict]:
    """AI Hub strategy.json 로드."""
    records = []
    for fpath in tqdm(sorted(label_dir.rglob('strategy.json')), desc='AI Hub labels'):
        match_id = fpath.parent.name
        with open(fpath, encoding='utf-8') as f:
            data = json.load(f)
        for entry in data.get('strategyArray', []):
            off = entry.get('offense_strategy', {})
            d1 = off.get('depth1', '')
            records.append({
                'match_id': match_id,
                'start_frame': entry.get('start', 0),
                'end_frame': entry.get('end', 0),
                'offense_tactic': d1,
                'offense_class': TACTIC_CLASSES.get(d1, -1),
                'offense_team': entry.get('offense_team', ''),
            })
    print(f'라벨: {len(records)}개')
    return records

aihub_labels = load_aihub_strategy_labels(AIHUB_LABEL_DIR) if AIHUB_LABEL_DIR.exists() else []


def normalize_positions(positions: np.ndarray, source: str,
                        img_w: float = 1920.0, img_h: float = 1080.0) -> np.ndarray:
    """FIX M2: 소스별 [0,1] 정규화. img_w/img_h는 JSON 메타데이터에서 전달."""
    pos = positions.copy().astype(np.float32)
    if source == 'aihub_image':
        pos[:, 0] /= img_w
        pos[:, 1] /= img_h
    elif source == 'soccernet':
        pos[:, 0] /= PITCH_X
        pos[:, 1] /= PITCH_Y
    else:
        for d in range(2):
            lo, hi = pos[:, d].min(), pos[:, d].max()
            if hi > lo:
                pos[:, d] = (pos[:, d] - lo) / (hi - lo)
    return np.clip(pos, 0.0, 1.0)


def load_aihub_image_labels(image_dir: Path, limit: int = 5000) -> list[dict]:
    """AI Hub 이미지 JSON에서 선수 좌표 추출. FIX M2: 이미지 해상도 메타데이터 사용."""
    frames = []
    for fpath in tqdm(sorted(image_dir.rglob('soccer_*.json'))[:limit], desc='AI Hub images'):
        with open(fpath, encoding='utf-8') as f:
            data = json.load(f)
        # FIX M2: 해상도를 메타데이터에서 읽기
        img_info = data.get('images', [{}])
        if isinstance(img_info, list) and img_info:
            img_info = img_info[0]
        img_w = float(img_info.get('width', 1920))
        img_h = float(img_info.get('height', 1080))

        positions, team_ids, roles = [], [], []
        ball_pos = None
        for ann in data.get('annotations', []):
            cat = ann.get('category', '')
            bbox = ann.get('bbox', [])
            if len(bbox) < 4:
                continue
            cx, cy = bbox[0] + bbox[2]/2, bbox[1] + bbox[3]/2
            if cat == 'ball':
                ball_pos = np.array([cx, cy], dtype=np.float32)
            elif 'player' in cat or 'goalkeeper' in cat:
                positions.append([cx, cy])
                team_ids.append(0 if 'team1' in cat else 1)
                roles.append('GK' if 'goalkeeper' in cat else 'FP')
        if len(positions) >= 10:
            raw = np.array(positions, dtype=np.float32)
            nums = re.findall(r'\d+', fpath.stem)
            frames.append({
                'frame_id': fpath.stem,
                'frame_num': int(nums[-1]) if nums else 0,
                'match_id': fpath.parent.name,
                'positions': normalize_positions(raw, 'aihub_image', img_w, img_h),
                'team_ids': np.array(team_ids, dtype=np.int64),
                'roles': roles,
                'ball_pos': normalize_positions(ball_pos.reshape(1,2), 'aihub_image', img_w, img_h).flatten() if ball_pos is not None else None,
                'source': 'aihub_image',
            })
    print(f'유효 프레임: {len(frames)}')
    return frames

aihub_frames = load_aihub_image_labels(AIHUB_IMAGE_DIR) if AIHUB_IMAGE_DIR.exists() else []

In [ ]:
import zipfile

def load_soccernet_tracking(tracking_dir: Path, max_games: int = 10, sample_rate: int = 25) -> list[dict]:
    """SoccerNet 트래킹 로드 (정규화 적용)."""
    frames = []
    def _proc(df, game_id):
        fc = next((c for c in df.columns if 'frame' in c.lower()), df.columns[0])
        for fid in sorted(df[fc].unique())[::sample_rate]:
            g = df[df[fc] == fid]
            xc = next((c for c in g.columns if c.lower() in ('x','pos_x','x_pos')), None)
            yc = next((c for c in g.columns if c.lower() in ('y','pos_y','y_pos')), None)
            tc = next((c for c in g.columns if 'team' in c.lower()), None)
            if xc and yc and len(g) >= 10:
                raw = g[[xc, yc]].values.astype(np.float32)
                tids = pd.factorize(g[tc].values)[0].astype(np.int64) if tc else np.zeros(len(g), dtype=np.int64)
                frames.append({
                    'frame_id': f'{game_id}_{int(fid)}', 'frame_num': int(fid),
                    'match_id': game_id,
                    'positions': normalize_positions(raw, 'soccernet'),
                    'team_ids': tids, 'ball_pos': None, 'roles': None, 'source': 'soccernet',
                })
    for zp in list(tracking_dir.glob('*.zip'))[:2]:
        with zipfile.ZipFile(zp) as zf:
            for cn in [n for n in zf.namelist() if n.endswith('.csv')][:max_games]:
                with zf.open(cn) as cf:
                    _proc(pd.read_csv(cf), Path(cn).stem)
    if not frames:
        for cp in list(tracking_dir.rglob('*.csv'))[:max_games]:
            _proc(pd.read_csv(cp), cp.stem)
    print(f'SoccerNet 프레임: {len(frames)}')
    return frames

sn_frames = load_soccernet_tracking(SOCCERNET_DIR) if (SOCCERNET_DIR.exists() and any(SOCCERNET_DIR.iterdir())) else []
if not sn_frames:
    print('SoccerNet 데이터 없음')

## 3. 그래프 변환 + 데이터셋 구축

In [ ]:
class TacticalData(Data):
    """FIX M3: PyG __inc__ 오버라이드로 배치 인덱스 자동 보정."""

    def __inc__(self, key, value, *args, **kwargs):
        if key == 'link_pairs':
            return self.x.size(0)
        if key == 'prev_edge_index':
            return self.prev_x.size(0) if hasattr(self, 'prev_x') else 0
        return super().__inc__(key, value, *args, **kwargs)


def frame_to_graph(
    positions: np.ndarray, team_ids: np.ndarray,
    ball_pos=None, prev_positions=None, roles=None, k_neighbors: int = 5,
) -> TacticalData:
    """프레임 → 그래프. 좌표는 [0,1] 정규화 완료 전제."""
    n = len(positions)
    k_neighbors = min(k_neighbors, n - 1)  # FIX: 선수 수 < k+1 가드

    vel = (positions - prev_positions) if (prev_positions is not None and len(prev_positions) == n) else np.zeros_like(positions)
    bdist = np.linalg.norm(positions - ball_pos, axis=1, keepdims=True) if ball_pos is not None else np.full((n,1), -1.0)
    pos_oh = np.array([POS_TO_ONEHOT.get(r, POS_TO_ONEHOT['FP']) for r in roles], dtype=np.float32) if roles else np.tile(POS_TO_ONEHOT['FP'], (n,1)).astype(np.float32)

    x = np.hstack([positions, vel, team_ids.reshape(-1,1), bdist, pos_oh]).astype(np.float32)

    dm = cdist(positions, positions)
    src, dst, ef = [], [], []
    for i in range(n):
        for j in np.argsort(dm[i])[1:k_neighbors+1]:
            src.append(i); dst.append(j)
            ef.append([dm[i,j], np.linalg.norm(vel[i]-vel[j]), float(team_ids[i]==team_ids[j]),
                       positions[j,0]-positions[i,0], positions[j,1]-positions[i,1]])

    return TacticalData(
        x=torch.tensor(x), edge_index=torch.tensor([src,dst], dtype=torch.long),
        edge_attr=torch.tensor(ef, dtype=torch.float32),
    )

# 검증
_g = frame_to_graph(np.random.rand(22,2).astype(np.float32), np.array([0]*11+[1]*11),
                    roles=['GK']+['DF']*4+['MF']*3+['FW']*3+['GK']+['DF']*4+['MF']*3+['FW']*3)
print(f'Graph: nodes={_g.num_nodes}, edges={_g.num_edges}, x={_g.x.shape}, edge={_g.edge_attr.shape}')

In [ ]:
def match_label(frame_num: int, match_id: str, labels: list[dict]) -> int:
    """범위 기반 라벨 매칭. 실패 시 -1."""
    for l in labels:
        if l['match_id'] == match_id and l['start_frame'] <= frame_num <= l['end_frame']:
            return l['offense_class']
    return -1


def compute_synergy_targets(
    positions: np.ndarray, team_ids: np.ndarray, roles: list[str] | None,
    n_samples: int = 20, seed: int = 0,
) -> tuple[torch.Tensor, torch.Tensor]:
    """FIX C2: 포지션 상보성 x 공간 근접도 기반 연속 시너지 타겟.

    Returns: (link_pairs [2k,2], link_labels [2k] in [0,1])
    """
    n = len(positions)
    if roles is None:
        roles = ['FP'] * n

    dm = cdist(positions, positions)
    max_dist = dm.max() + 1e-8

    rng = np.random.default_rng(seed)
    # 같은 팀 쌍 + 다른 팀 쌍 균등 샘플
    same_pairs = [(a,b) for a in range(n) for b in range(a+1,n) if team_ids[a]==team_ids[b]]
    diff_pairs = [(a,b) for a in range(n) for b in range(a+1,n) if team_ids[a]!=team_ids[b]]

    k = min(len(same_pairs), len(diff_pairs), n_samples)
    if k == 0:
        return torch.zeros((0,2), dtype=torch.long), torch.zeros(0)

    sel_same = [same_pairs[i] for i in rng.choice(len(same_pairs), k, replace=False)]
    sel_diff = [diff_pairs[i] for i in rng.choice(len(diff_pairs), k, replace=False)]

    pairs, labels = [], []
    for a, b in sel_same:
        comp = COMPLEMENTARITY[POS_TO_IDX.get(roles[a],'FP'), POS_TO_IDX.get(roles[b],'FP')]
        prox = max(0, 1.0 - dm[a,b] / max_dist)
        pairs.append([a,b])
        labels.append(float(comp * prox))  # [0, 1] 연속값

    for a, b in sel_diff:
        pairs.append([a,b])
        labels.append(0.0)  # 다른 팀 = 시너지 0

    return torch.tensor(pairs, dtype=torch.long), torch.tensor(labels, dtype=torch.float32)


# --- 포메이션 템플릿 ---
FORMATIONS = {
    '433': (np.array([[.05,.50],[.20,.15],[.20,.38],[.20,.62],[.20,.85],
                      [.40,.25],[.40,.50],[.40,.75],[.60,.15],[.60,.50],[.60,.85]], dtype=np.float32),
            ['GK']+['DF']*4+['MF']*3+['FW']*3),
    '442': (np.array([[.05,.50],[.20,.15],[.20,.38],[.20,.62],[.20,.85],
                      [.40,.15],[.40,.38],[.40,.62],[.40,.85],[.60,.35],[.60,.65]], dtype=np.float32),
            ['GK']+['DF']*4+['MF']*4+['FW']*2),
    '352': (np.array([[.05,.50],[.20,.25],[.20,.50],[.20,.75],
                      [.35,.10],[.40,.30],[.40,.50],[.40,.70],[.35,.90],[.60,.35],[.60,.65]], dtype=np.float32),
            ['GK']+['DF']*3+['MF']*5+['FW']*2),
    '4231': (np.array([[.05,.50],[.20,.15],[.20,.38],[.20,.62],[.20,.85],
                       [.35,.35],[.35,.65],[.50,.15],[.50,.50],[.50,.85],[.65,.50]], dtype=np.float32),
             ['GK']+['DF']*4+['MF']*5+['FW']*1),
}

# FIX H3: 포메이션 ↔ 전술 상관관계 매핑
FORMATION_TACTIC_MAP = {
    '433': ['빌드업-크로스형', '빌드업-측면침투형', '공격전환'],
    '442': ['수비-중위압박', '수비-저위블록', '수비전환'],
    '352': ['빌드업-중앙침투형', '빌드업-롱볼형', '수비-고위압박'],
    '4231': ['빌드업-중앙침투형', '세트피스-코너킥', '세트피스-프리킥'],
}

In [ ]:
def build_dataset(frames: list[dict], labels: list[dict] | None = None) -> list[TacticalData]:
    """프레임 → TacticalData 리스트.

    FIX C1: 각 그래프에 이전 프레임의 그래프 구조(prev_x, prev_edge_*)를 함께 저장.
    FIX C3: 라벨 매칭 실패 → y_mask=0 (Head B loss 마스킹).
    """
    graphs = []
    matched, unmatched = 0, 0
    prev_graph = None
    prev_match = None

    for i, frame in enumerate(tqdm(frames, desc='Building graphs')):
        n = len(frame['positions'])
        prev_pos = frames[i-1]['positions'] if (i > 0 and len(frames[i-1]['positions']) == n) else None

        g = frame_to_graph(
            frame['positions'], frame['team_ids'],
            frame.get('ball_pos'), prev_pos, frame.get('roles'),
        )

        # --- 전술 라벨 + FIX C3: 매칭 실패 마스킹 ---
        tc = -1
        if labels:
            tc = match_label(frame.get('frame_num', 0), frame.get('match_id', ''), labels)
            if tc >= 0:
                matched += 1
            else:
                unmatched += 1

        if tc >= 0:
            g.y = torch.tensor([tc], dtype=torch.long)
            g.y_mask = torch.tensor([1.0])  # loss 적용
        else:
            g.y = torch.tensor([0], dtype=torch.long)  # placeholder
            g.y_mask = torch.tensor([0.0])  # FIX C3: loss 마스킹

        g.change_label = torch.tensor([0.0])

        # --- 시너지 타겟 (FIX C2) ---
        lp, ll = compute_synergy_targets(
            frame['positions'], frame['team_ids'], frame.get('roles'), seed=i,
        )
        g.link_pairs = lp
        g.link_labels = ll

        # --- FIX C1: 이전 프레임 그래프 저장 (temporal differencing) ---
        same_match = (prev_match == frame.get('match_id'))
        if prev_graph is not None and same_match:
            g.prev_x = prev_graph.x.clone()
            g.prev_edge_index = prev_graph.edge_index.clone()
            g.prev_edge_attr = prev_graph.edge_attr.clone()
            g.prev_num_nodes = torch.tensor([prev_graph.num_nodes])
            g.has_prev = torch.tensor([1.0])
        else:
            # 첫 프레임: self-reference (Head C loss 마스킹)
            g.prev_x = g.x.clone()
            g.prev_edge_index = g.edge_index.clone()
            g.prev_edge_attr = g.edge_attr.clone()
            g.prev_num_nodes = torch.tensor([g.num_nodes])
            g.has_prev = torch.tensor([0.0])

        prev_graph = g
        prev_match = frame.get('match_id')
        graphs.append(g)

    # 변화점 라벨 (연속 프레임 간 전술 변경)
    ch = 0
    for i in range(1, len(graphs)):
        if graphs[i].y_mask.item() > 0 and graphs[i-1].y_mask.item() > 0:
            if graphs[i].y.item() != graphs[i-1].y.item():
                graphs[i].change_label = torch.tensor([1.0])
                ch += 1

    print(f'데이터셋: {len(graphs)}개')
    if labels:
        print(f'  라벨 매칭: {matched} 성공, {unmatched} 실패(마스킹)')
    print(f'  변화점: {ch}개 ({ch/max(len(graphs),1)*100:.1f}%)')
    return graphs

In [ ]:
# --- 실제 데이터 or 가상 데이터 ---
all_frames = (aihub_frames or []) + (sn_frames or [])

if not all_frames:
    print('실제 데이터 없음 → 포메이션-전술 연관 가상 데이터 생성')
    print('  (구조 검증 전용 — 성능 수치는 실 데이터 기준 아님)\n')

    aihub_labels = []
    rng = np.random.default_rng(42)

    for i in range(1200):
        match_id = f'synth_{i // 100}'
        fm_name = list(FORMATIONS.keys())[i % len(FORMATIONS)]
        fm_pos, fm_roles = FORMATIONS[fm_name]

        # FIX H3: 포메이션에 맞는 전술 선택 (상관관계 부여)
        tactic = rng.choice(FORMATION_TACTIC_MAP[fm_name])
        tc = TACTIC_CLASSES[tactic]

        team_a = np.clip(fm_pos + rng.normal(0, 0.03, fm_pos.shape).astype(np.float32), 0, 1)
        fm_b_name = list(FORMATIONS.keys())[(i+1) % len(FORMATIONS)]
        fm_b_pos, fm_b_roles = FORMATIONS[fm_b_name]
        team_b = np.clip(1.0 - fm_b_pos + rng.normal(0, 0.03, fm_b_pos.shape).astype(np.float32), 0, 1)

        all_frames.append({
            'frame_id': f'synth_{i}', 'frame_num': i, 'match_id': match_id,
            'positions': np.vstack([team_a, team_b]),
            'team_ids': np.array([0]*11 + [1]*11, dtype=np.int64),
            'ball_pos': rng.random(2).astype(np.float32) * 0.8 + 0.1,
            'roles': fm_roles + fm_b_roles,
            'source': 'synthetic',
        })
        aihub_labels.append({
            'match_id': match_id, 'start_frame': i, 'end_frame': i,
            'offense_tactic': tactic, 'offense_class': tc,
        })

    print(f'가상 프레임: {len(all_frames)}, 라벨: {len(aihub_labels)}')

graphs = build_dataset(all_frames, aihub_labels if aihub_labels else None)

In [ ]:
# Train / Val / Test
train_g, test_g = train_test_split(graphs, test_size=0.15, random_state=42)
train_g, val_g = train_test_split(train_g, test_size=0.12, random_state=42)
print(f'Train: {len(train_g)}, Val: {len(val_g)}, Test: {len(test_g)}')

BS = 32
train_loader = DataLoader(train_g, batch_size=BS, shuffle=True)
val_loader = DataLoader(val_g, batch_size=BS)
test_loader = DataLoader(test_g, batch_size=BS)

## 4. 모델 정의

```
Input (N, 11)
  │
  ├─ BatchNorm1d ──────────────────────────┐
  ├─ GATv2Conv L1 (11→64, h=4) → 256      │ Residual
  ├─ GATv2Conv L2 (256→64, h=1) → 64  ←───┘ (proj 11→64)
  │
  ├─ mean_pool + max_pool → 128 (graph_emb)
  │
  ├─ Adapter A → Head A (Link Pred, MSE)
  ├─ Adapter B → Head B (Tactic, CE)
  └─ Adapter C + cat([emb_t, emb_{t-1}, diff]) → Head C (Change, BCE)
```

In [ ]:
class SharedGATBackbone(nn.Module):
    """FIX M5: 2-layer + residual. FIX H1: mean+max pool."""

    def __init__(self, in_dim: int, hidden_dim: int, heads: int = 4, edge_dim: int = 5):
        super().__init__()
        self.input_norm = nn.BatchNorm1d(in_dim)
        self.conv1 = GATv2Conv(in_dim, hidden_dim, heads=heads, edge_dim=edge_dim, concat=True)
        self.norm1 = nn.LayerNorm(hidden_dim * heads)
        self.conv2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, edge_dim=edge_dim, concat=False)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.res_proj = nn.Linear(in_dim, hidden_dim)

    def forward(self, x, edge_index, edge_attr, batch):
        x0 = self.input_norm(x)
        h = F.elu(self.norm1(self.conv1(x0, edge_index, edge_attr)))
        h = F.dropout(h, p=0.1, training=self.training)
        h = F.elu(self.norm2(self.conv2(h, edge_index, edge_attr)))
        node_emb = h + self.res_proj(x0)
        g_mean = global_mean_pool(node_emb, batch)
        g_max = global_max_pool(node_emb, batch)
        graph_emb = torch.cat([g_mean, g_max], dim=-1)
        return node_emb, graph_emb


class TaskAdapter(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU(), nn.LayerNorm(out_dim))
    def forward(self, x):
        return self.net(x)


class LearnableComplementarity(nn.Module):
    """학습 가능한 포지션 상보성 행렬 (하이브리드).

    도메인 초기값으로 시작 → GNN 학습에서 데이터 기반 정밀화.
    Synergy(i,j) = sigmoid(C_learned[pos_i, pos_j]) * exp(-distance(i,j))
    """

    def __init__(self, init_matrix: np.ndarray):
        super().__init__()
        init_t = torch.tensor(init_matrix, dtype=torch.float32).clamp(0.01, 0.99)
        init_logits = torch.log(init_t / (1.0 - init_t))
        self.C_logits = nn.Parameter(init_logits)

    def forward(self, pos_idx_i: torch.Tensor, pos_idx_j: torch.Tensor,
                distance: torch.Tensor) -> torch.Tensor:
        C = torch.sigmoid(self.C_logits)
        comp = C[pos_idx_i, pos_idx_j]
        proximity = torch.exp(-distance)
        return comp * proximity

    def get_matrix(self) -> np.ndarray:
        with torch.no_grad():
            return torch.sigmoid(self.C_logits).cpu().numpy()


class LinkPredictionHead(nn.Module):
    """Head A: 학습 가능 상보성 행렬 + GNN 임베딩 시너지."""
    def __init__(self, emb_dim: int, init_matrix: np.ndarray):
        super().__init__()
        self.complementarity = LearnableComplementarity(init_matrix)
        self.adapter = TaskAdapter(emb_dim, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, emb_dim), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(emb_dim, emb_dim // 2), nn.GELU(), nn.Linear(emb_dim // 2, 1),
        )

    def forward(self, node_emb, pairs, pos_indices=None, distances=None):
        if pairs.numel() == 0:
            return torch.zeros(0, device=node_emb.device)
        adapted = self.adapter(node_emb)
        h_i, h_j = adapted[pairs[:, 0]], adapted[pairs[:, 1]]
        emb_score = torch.sigmoid(self.mlp(torch.cat([h_i, h_j], dim=-1))).squeeze(-1)

        if pos_indices is not None and distances is not None:
            pi = pos_indices[pairs[:, 0]]
            pj = pos_indices[pairs[:, 1]]
            comp_score = self.complementarity(pi, pj, distances)
            return 0.5 * emb_score + 0.5 * comp_score
        return emb_score


class TacticClassificationHead(nn.Module):
    def __init__(self, graph_dim: int, num_classes: int):
        super().__init__()
        self.adapter = TaskAdapter(graph_dim, graph_dim // 2)
        self.mlp = nn.Sequential(
            nn.Linear(graph_dim // 2, graph_dim // 2), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(graph_dim // 2, num_classes),
        )
    def forward(self, graph_emb):
        return self.mlp(self.adapter(graph_emb))


class ChangeDetectionHead(nn.Module):
    def __init__(self, graph_dim: int):
        super().__init__()
        self.adapter = TaskAdapter(graph_dim * 3, graph_dim)
        self.mlp = nn.Sequential(
            nn.Linear(graph_dim, graph_dim // 2), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(graph_dim // 2, 1),
        )
    def forward(self, emb_t, emb_prev):
        diff = emb_t - emb_prev
        return self.mlp(self.adapter(torch.cat([emb_t, emb_prev, diff], dim=-1)))


class UncertaintyWeighting(nn.Module):
    def __init__(self, n_tasks: int):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(n_tasks))
    def forward(self, *losses):
        total = torch.tensor(0.0, device=self.log_vars.device)
        for i, loss in enumerate(losses):
            precision = torch.exp(-self.log_vars[i])
            total = total + precision * loss + self.log_vars[i]
        return total


class TacticalGNN(nn.Module):
    """공유 GAT 백본 + 3-Head + 학습 가능 상보성 행렬."""

    def __init__(self, node_in_dim=11, hidden_dim=64, num_classes=13,
                 gat_heads=4, edge_dim=5, complementarity_init=None):
        super().__init__()
        self.hidden_dim = hidden_dim
        graph_dim = hidden_dim * 2

        self.backbone = SharedGATBackbone(node_in_dim, hidden_dim, gat_heads, edge_dim)
        self.head_link = LinkPredictionHead(
            hidden_dim,
            complementarity_init if complementarity_init is not None else COMPLEMENTARITY_INIT,
        )
        self.head_tactic = TacticClassificationHead(graph_dim, num_classes)
        self.head_change = ChangeDetectionHead(graph_dim)
        self.uncertainty = UncertaintyWeighting(3)

    def _make_prev_batch(self, prev_num_nodes: torch.Tensor) -> torch.Tensor:
        pnn = prev_num_nodes.view(-1)  # FIX: 0-d 텐서 방지 → 항상 1-d
        return torch.repeat_interleave(
            torch.arange(pnn.size(0), device=pnn.device), pnn,
        )

    def forward(self, data):
        node_emb, graph_emb = self.backbone(
            data.x, data.edge_index, data.edge_attr, data.batch,
        )

        prev_batch = self._make_prev_batch(data.prev_num_nodes)
        _, prev_graph_emb = self.backbone(
            data.prev_x, data.prev_edge_index, data.prev_edge_attr, prev_batch,
        )

        tactic_logits = self.head_tactic(graph_emb)
        change_logits = self.head_change(graph_emb, prev_graph_emb)

        link_scores = torch.zeros(0, device=node_emb.device)
        if hasattr(data, 'link_pairs') and data.link_pairs.numel() > 0:
            pos_indices = data.x[:, 6:11].argmax(dim=1)
            pairs = data.link_pairs
            dists = torch.norm(data.x[pairs[:, 0], :2] - data.x[pairs[:, 1], :2], dim=1)
            link_scores = self.head_link(node_emb, pairs, pos_indices, dists)

        return {
            'node_emb': node_emb, 'graph_emb': graph_emb,
            'tactic_logits': tactic_logits,
            'change_logits': change_logits,
            'link_scores': link_scores,
        }


model = TacticalGNN(
    node_in_dim=NODE_IN_DIM, hidden_dim=HIDDEN_DIM,
    num_classes=NUM_TACTIC_CLASSES, edge_dim=EDGE_IN_DIM,
    complementarity_init=COMPLEMENTARITY_INIT,
).to(device)

print(f'TacticalGNN: {sum(p.numel() for p in model.parameters()):,} params')
for name, mod in [('Backbone', model.backbone), ('Head A', model.head_link),
                  ('  └ Complementarity', model.head_link.complementarity),
                  ('Head B', model.head_tactic), ('Head C', model.head_change),
                  ('Uncertainty', model.uncertainty)]:
    print(f'  {name:20s}: {sum(p.numel() for p in mod.parameters()):,}')

print(f'\n초기 상보성 행렬 (학습 전):')
print(model.head_link.complementarity.get_matrix().round(2))

## 5. 학습

In [ ]:
# 변화점 pos_weight
cr = sum(g.change_label.item() for g in graphs) / max(len(graphs), 1)
cpw = max((1.0 - cr) / max(cr, 0.01), 1.0)
print(f'변화점 비율: {cr:.3f}, pos_weight: {cpw:.1f}')

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60, eta_min=1e-5)

criterion_tactic = nn.CrossEntropyLoss(reduction='none')
criterion_link = nn.MSELoss()
criterion_change = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([cpw], device=device), reduction='none',
)


def train_epoch(loader):
    model.train()
    stats = {'loss': 0, 'tac_correct': 0, 'tac_total': 0, 'link_mse': 0, 'link_n': 0}

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)

        y = batch.y.view(-1)
        mask_b = batch.y_mask.view(-1)
        ch_label = batch.change_label.view(-1)
        mask_c = batch.has_prev.view(-1)

        # Head B
        tac_loss_all = criterion_tactic(out['tactic_logits'], y)
        loss_tactic = (tac_loss_all * mask_b).sum() / mask_b.sum().clamp(min=1)

        # Head C
        ch_loss_all = criterion_change(out['change_logits'].view(-1), ch_label)
        loss_change = (ch_loss_all * mask_c).sum() / mask_c.sum().clamp(min=1)

        # Head A
        loss_link = torch.tensor(0.0, device=device)
        if out['link_scores'].numel() > 0:
            loss_link = criterion_link(out['link_scores'], batch.link_labels.to(device))
            stats['link_mse'] += loss_link.item() * len(out['link_scores'])
            stats['link_n'] += len(out['link_scores'])

        loss = model.uncertainty(loss_tactic, loss_link, loss_change)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        stats['loss'] += loss.item() * batch.num_graphs
        valid = mask_b > 0
        if valid.any():
            pred = out['tactic_logits'][valid].argmax(dim=1)
            stats['tac_correct'] += (pred == y[valid]).sum().item()
            stats['tac_total'] += valid.sum().item()

    n = max(stats['tac_total'], 1)
    return (
        stats['loss'] / max(len(loader.dataset), 1),
        stats['tac_correct'] / n,
        stats['link_mse'] / max(stats['link_n'], 1),
    )


@torch.no_grad()
def eval_epoch(loader):
    model.eval()
    stats = {'loss': 0, 'tac_correct': 0, 'tac_total': 0, 'link_mse': 0, 'link_n': 0}

    for batch in loader:
        batch = batch.to(device)
        out = model(batch)

        y = batch.y.view(-1)
        mask_b = batch.y_mask.view(-1)
        ch_label = batch.change_label.view(-1)
        mask_c = batch.has_prev.view(-1)

        tac_loss_all = criterion_tactic(out['tactic_logits'], y)
        loss_tactic = (tac_loss_all * mask_b).sum() / mask_b.sum().clamp(min=1)

        ch_loss_all = criterion_change(out['change_logits'].view(-1), ch_label)
        loss_change = (ch_loss_all * mask_c).sum() / mask_c.sum().clamp(min=1)

        loss_link = torch.tensor(0.0, device=device)
        if out['link_scores'].numel() > 0:
            loss_link = criterion_link(out['link_scores'], batch.link_labels.to(device))
            stats['link_mse'] += loss_link.item() * len(out['link_scores'])
            stats['link_n'] += len(out['link_scores'])

        loss = model.uncertainty(loss_tactic, loss_link, loss_change)
        stats['loss'] += loss.item() * batch.num_graphs

        valid = mask_b > 0
        if valid.any():
            pred = out['tactic_logits'][valid].argmax(dim=1)
            stats['tac_correct'] += (pred == y[valid]).sum().item()
            stats['tac_total'] += valid.sum().item()

    n = max(stats['tac_total'], 1)
    return (
        stats['loss'] / max(len(loader.dataset), 1),
        stats['tac_correct'] / n,
        stats['link_mse'] / max(stats['link_n'], 1),
    )

In [ ]:
EPOCHS = 60
best_val_acc = 0.0
history = {k: [] for k in ['t_loss','v_loss','t_tac','v_tac','t_link','v_link']}

print('=== Training (Uncertainty Weighting) ===\n')

for epoch in range(1, EPOCHS + 1):
    tl, tt, tlk = train_epoch(train_loader)
    vl, vt, vlk = eval_epoch(val_loader)
    scheduler.step()

    history['t_loss'].append(tl); history['v_loss'].append(vl)
    history['t_tac'].append(tt); history['v_tac'].append(vt)
    history['t_link'].append(tlk); history['v_link'].append(vlk)

    if vt > best_val_acc:
        best_val_acc = vt
        torch.save(model.state_dict(), str(MODEL_DIR / 'best_v3.pt'))

    if epoch % 5 == 0 or epoch == 1:
        uw = torch.exp(-model.uncertainty.log_vars).detach().cpu().numpy()
        print(f'Epoch {epoch:3d} | Loss {tl:.4f}/{vl:.4f} | '
              f'Tac {tt:.3f}/{vt:.3f} | Link MSE {tlk:.4f}/{vlk:.4f} | '
              f'UW [{uw[0]:.2f},{uw[1]:.2f},{uw[2]:.2f}] | Best {best_val_acc:.3f}')

print(f'\nComplete. Best Val Acc: {best_val_acc:.3f}')

## 6. 평가

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history['t_loss'], label='Train'); axes[0].plot(history['v_loss'], label='Val')
axes[0].set_title('Loss (Uncertainty Weighted)'); axes[0].legend()
axes[1].plot(history['t_tac'], label='Train'); axes[1].plot(history['v_tac'], label='Val')
axes[1].set_title('Head B: Tactic Acc'); axes[1].legend()
axes[2].plot(history['t_link'], label='Train'); axes[2].plot(history['v_link'], label='Val')
axes[2].set_title('Head A: Link MSE'); axes[2].legend()
plt.tight_layout(); plt.savefig(str(PROJECT_DIR / 'curves_v3.png'), dpi=150); plt.show()

In [ ]:
model.load_state_dict(torch.load(str(MODEL_DIR / 'best_v3.pt')))
tl, ta, tlk = eval_epoch(test_loader)
print(f'=== Test === Loss: {tl:.4f}, Tactic Acc: {ta:.3f}, Link MSE: {tlk:.4f}')

preds, labels = [], []
model.eval()
with torch.no_grad():
    for b in test_loader:
        b = b.to(device)
        o = model(b)
        m = b.y_mask.view(-1) > 0
        if m.any():
            preds.extend(o['tactic_logits'][m].argmax(1).cpu().numpy())
            labels.extend(b.y.view(-1)[m].cpu().numpy())

pc = sorted(set(labels) | set(preds))
print('\n=== Tactic Classification ===')
print(classification_report(labels, preds, labels=pc,
      target_names=[IDX_TO_TACTIC.get(i,f'c{i}') for i in pc], zero_division=0))

## 7. 추론 시뮬레이션

### 7-A. AdaptFit 시너지 (Head A)

In [ ]:
model.eval()
print('=== AdaptFit 시너지 (포지션 상보성 기반) ===')
print('시나리오: 4-3-3 팀에 FW 후보 추가\n')

fm_pos, fm_roles = FORMATIONS['433']
fm_b_pos, fm_b_roles = FORMATIONS['442']
team_b = 1.0 - fm_b_pos.copy()
cand = np.array([[0.55, 0.42]], dtype=np.float32)

all_pos = np.vstack([fm_pos, team_b, cand])
all_teams = np.array([0]*11+[1]*11+[0], dtype=np.int64)
all_roles = fm_roles + fm_b_roles + ['FW']

g = frame_to_graph(all_pos, all_teams, np.array([0.5,0.5]), roles=all_roles)
# 더미 prev (self-reference)
g.prev_x = g.x.clone(); g.prev_edge_index = g.edge_index.clone()
g.prev_edge_attr = g.edge_attr.clone(); g.prev_num_nodes = torch.tensor([g.num_nodes])
g.has_prev = torch.tensor([0.0]); g.y = torch.tensor([0]); g.y_mask = torch.tensor([0.0])
g.change_label = torch.tensor([0.0])
g.link_pairs = torch.zeros((0,2), dtype=torch.long); g.link_labels = torch.zeros(0)

batch = Batch.from_data_list([g]).to(device)
with torch.no_grad():
    o = model(batch)
    cand_idx = 22
    for i in range(11):
        pairs = torch.tensor([[cand_idx, i]], dtype=torch.long, device=device)
        score = model.head_link(o['node_emb'], pairs).item()
        w = POSITION_WEIGHTS[fm_roles[i]]
        comp = COMPLEMENTARITY[POS_TO_IDX['FW'], POS_TO_IDX[fm_roles[i]]]
        print(f'  Player {i:2d} ({fm_roles[i]}) | synergy: {score:.3f} | '
              f'comp: {comp:.1f} | weight: {w} | final: {score*w:.3f}')

### 7-B. La Paz 전술 + 변화점 (Head B + C)

In [ ]:
print('=== La Paz 전술 분석 + 변화점 감지 ===')
print('(테스트 20프레임 — Head C는 이전 프레임과 비교)\n')

prev_tactic = None
for i in range(min(20, len(test_g))):
    sample = Batch.from_data_list([test_g[i]]).to(device)
    t0 = time.time()
    with torch.no_grad():
        o = model(sample)
        pc = o['tactic_logits'].argmax(1).item()
        conf = F.softmax(o['tactic_logits'], dim=1).max().item()
        chp = torch.sigmoid(o['change_logits']).item()
    ms = (time.time() - t0) * 1000

    tn = IDX_TO_TACTIC.get(pc, f'c{pc}')
    has_p = test_g[i].has_prev.item() > 0
    changed = prev_tactic is not None and pc != prev_tactic
    mk = ' << CHANGE' if changed else ''

    print(f'Frame {i:3d} | {tn:20s} | conf: {conf:.2f} | '
          f'change_p: {chp:.2f}{" (temporal)" if has_p else " (no prev)"} | {ms:.1f}ms{mk}')
    if changed:
        print(f'          -> {IDX_TO_TACTIC.get(prev_tactic,"?")} → {tn}')
    prev_tactic = pc

## 8. 모델 저장

In [ ]:
uw = torch.exp(-model.uncertainty.log_vars).detach().cpu().numpy()
learned_comp = model.head_link.complementarity.get_matrix()

ckpt = {
    'model_state_dict': model.state_dict(),
    'config': {'node_in_dim': NODE_IN_DIM, 'edge_dim': EDGE_IN_DIM,
               'hidden_dim': HIDDEN_DIM, 'num_classes': NUM_TACTIC_CLASSES, 'gat_heads': 4},
    'tactic_classes': TACTIC_CLASSES, 'idx_to_tactic': IDX_TO_TACTIC,
    'position_weights': POSITION_WEIGHTS,
    'complementarity_init': COMPLEMENTARITY_INIT.tolist(),
    'complementarity_learned': learned_comp.tolist(),
    'learned_task_weights': {'tactic': uw[0], 'link': uw[1], 'change': uw[2]},
    'history': history, 'best_val_acc': best_val_acc,
}
torch.save(ckpt, str(MODEL_DIR / 'tactical_gnn_v3.pt'))
torch.save(model.backbone.state_dict(), str(MODEL_DIR / 'shared_backbone_v3.pt'))

print(f'모델 저장 완료')
print(f'  Backbone:  {sum(p.numel() for p in model.backbone.parameters()):,}')
print(f'  Head A:    {sum(p.numel() for p in model.head_link.parameters()):,}')
print(f'  Head B:    {sum(p.numel() for p in model.head_tactic.parameters()):,}')
print(f'  Head C:    {sum(p.numel() for p in model.head_change.parameters()):,}')
print(f'  Total:     {sum(p.numel() for p in model.parameters()):,}')
print(f'  Task weights: tactic={uw[0]:.3f}, link={uw[1]:.3f}, change={uw[2]:.3f}')

# 학습 전/후 상보성 행렬 비교
print(f'\n=== 상보성 행렬 비교 (도메인 초기값 → 학습 후) ===')
print(f'포지션:  {POSITION_NAMES}')
print(f'\n초기값 (도메인 지식):')
print(COMPLEMENTARITY_INIT.round(2))
print(f'\n학습 후 (데이터 정밀화):')
print(learned_comp.round(3))
print(f'\n차이 (학습이 발견한 보정):')
diff = learned_comp - COMPLEMENTARITY_INIT
print(np.array2string(diff, precision=3, sign='+'))

---
## 특허 청구항 ↔ 코드 매핑

| 청구항 | 구현 | v3 차별점 |
|--------|------|----------|
| 1: 그래프→GAT→시너지 | `frame_to_graph` → `SharedGATBackbone` → `LinkPredictionHead` | 포지션 상보성 행렬 기반 연속 시너지 |
| 2: GAT 어텐션 | `GATv2Conv` x2 + residual + BatchNorm | 2-layer + skip connection |
| 3: 링크예측 MLP | `LinkPredictionHead` (adapter + concat + sigmoid) | TaskAdapter로 태스크 특화 |
| 6: 포지션 가중치 | 원핫 인코딩(학습) + `POSITION_WEIGHTS`(추론) | `COMPLEMENTARITY` 행렬 |
| NEW: 시간적 변화점 | `ChangeDetectionHead(emb_t, emb_prev)` | temporal differencing |
| NEW: 동적 태스크 밸런싱 | `UncertaintyWeighting` (Kendall 2018) | 학습 가능 가중치 |

## v2 → v3 전체 변경 이력

| # | 이슈 | 수정 |
|---|------|------|
| C1 | Head C 단일프레임 | `cat([emb_t, emb_{t-1}, diff])` temporal differencing |
| C2 | Head A 팀소속 분류 | `COMPLEMENTARITY` × 공간근접도 연속타겟 + MSELoss |
| C3 | 라벨 실패→랜덤 | `y_mask` Head B loss 마스킹 |
| H1 | mean_pool 정보손실 | mean + max 결합 (graph_emb 2D) |
| H2 | Negative transfer | `TaskAdapter` per head |
| H3 | 가상데이터 무의미 | `FORMATION_TACTIC_MAP` 상관관계 |
| M1 | 정적 loss 가중치 | `UncertaintyWeighting` (학습 가능) |
| M2 | 해상도 하드코딩 | JSON `images.width/height` 사용 |
| M3 | 콜레이터 충돌 | `TacticalData.__inc__()` 오버라이드 |
| M4 | lr 과다 | lr=3e-4, eta_min=1e-5 |
| M5 | 3-layer over-smoothing | 2-layer + residual |